# Imports

In [1]:
# Enable autoreloading, delete this for deploying!!!
%load_ext autoreload
%autoreload 2
%load_ext memory_profiler

In [2]:
%%time
import os  # Provides functions to interact with the operating system


# NOTE: UNCOMMENT IF NO GPU AVAILABLE!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Adjust device IDs as per your setup,

# IBMS Libraries
from scToolkit import sc_code
from scToolkit import sc_plots
from scToolkit import sc_utils


# Some libraries (e.g., decoupler) may require setting rarely used parallelization
# backends. This function ensures all common thread settings are configured.
# utils.set_threads(10)


/ssd/projects/Proximap/mambaforge/scanpy_stable_25_04_latest/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


CUDA devices visible: 0
CPU times: user 5.65 s, sys: 511 ms, total: 6.16 s
Wall time: 4.81 s


In [3]:
# Other general libraries that might be hepflull for analysis

import re  # Enables regular expression matching and text processing
import time  # Offers time-related functions (e.g., sleep, timestamps)
# in Python, most assignments are references (symbolic links), not copies
from copy import deepcopy as copy  # Creates deep copies of complex objects
import scanpy as sc  # Toolkit for analyzing single-cell gene expression data
import anndata as ad  # Provides AnnData structure for annotated data matrices
import seaborn as sns  # Simplifies creation of attractive statistical plots
import numpy as np  # Supports numerical operations and N-dimensional arrays
import pandas as pd  # Offers data structures and tools for data analysis
import matplotlib.pyplot as plt  # Enables plotting and figure customization
from itertools import combinations  # Gene∫Ωrates all k-length combinations of an iterable
import scipy.stats as stats  # Contains statistical distributions and test functions


# Define rcParams for customization, notebook showing is figure.dpi, reduce to not overload the nootebook sizes
plt.rcParams.update({
    # "figure.figsize": (8, 6),  # Figure size in inches
    "savefig.dpi": 400,       # Dots per inch for saved figures
    "figure.dpi": 100,       # Dots per inch for saved figures
    #"axes.titlesize": 16,     # Title font size
    #"axes.labelsize": 14,     # Label font size
    #"xtick.labelsize": 12,    # X-axis tick label size
    #"ytick.labelsize": 12     # Y-axis tick label size
})

# Setup matplotlib notebook backend
%matplotlib inline


In [4]:
start = time.time()

# Combine into one Adata object

In [5]:
blood_adata = sc_code.load_h5ad("../data/disco_blood_v1.1.h5ad")

In [6]:
tonsil_adata = sc_code.load_h5ad("../data/disco_tonsil_v01.h5ad")

In [7]:
# Subsetting the DISCO tonsil to non-Tonsil Atlas samples only
tonsil_adata = tonsil_adata[tonsil_adata.obs["projectid"].isin(
    ['GSE165860', 'GSE139324', 'GSE159674', 'GSE119507',
     'GSE139891', 'GSE115006', 'GSE119506'])]
tonsil_adata

View of AnnData object with n_obs × n_vars = 92114 × 32577
    obs: 'orig.ident', 'ncount_rna', 'nfeature_rna', 'sample', 'sample_id', 'projectid', 'sample_type', 'tissue', 'platform', 'cell_sorting', 'subjectid', 'age', 'gender', 'processstatus', 'rna_snn_res.0.8', 'seurat_clusters', 'percent.mt', 'percent.rp', 'ct', 'n_genes', 'n_counts', 'pct_MT', 'pct_RBS', 'pct_MTRBS', 'pct_PSEUDO', 'pct_HG', 'pct_gtex', 'pct_hpa', 'pct_hrt', 'pct_cellminer', 'pct_klijn', 'pct_IG_C_gene', 'pct_IG_C_pseudogene', 'pct_IG_J_gene', 'pct_IG_V_gene', 'pct_IG_V_pseudogene', 'pct_TR_C_gene', 'pct_TR_J_gene', 'pct_TR_V_gene', 'pct_TR_V_pseudogene', 'pct_lncRNA', 'pct_miRNA', 'pct_protein_coding', 'pct_ribozyme', 'pct_snRNA', 'pct_snoRNA', 'pct_pseudogenes_hg', 'n_unique'
    var: 'gene', 'MT', 'RBS', 'MTRBS', 'PSEUDO', 'HG', 'gtex', 'hpa', 'hrt', 'cellminer', 'klijn', 'IG_C_gene', 'IG_C_pseudogene', 'IG_D_gene', 'IG_J_gene', 'IG_J_pseudogene', 'IG_V_gene', 'IG_V_pseudogene', 'TR_C_gene', 'TR_D_gene', 'TR_J

In [8]:
tonsil_atlas_adata = sc_code.load_h5ad("../data/tonsil_sc.h5ad")
tonsil_atlas_adata

AnnData object with n_obs × n_vars = 462352 × 37378
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'barcode', 'donor_id', 'gem_id', 'library_name', 'assay', 'sex', 'age', 'age_group', 'hospital', 'cohort_type', 'cause_for_tonsillectomy', 'is_hashed', 'preservation', 'pct_mt', 'pct_ribosomal', 'pDNN_hashing', 'pDNN_scrublet', 'pDNN_union', 'scrublet_doublet_scores', 'S.Score', 'G2M.Score', 'Phase', 'scrublet_predicted_doublet', 'doublet_score_scDblFinder', 'annotation_level_1', 'annotation_level_1_probability', 'annotation_figure_1', 'annotation_20220215', 'annotation_20220619', 'annotation_20230508', 'annotation_20230508_probability', 'UMAP_1_level_1', 'UMAP_2_level_1', 'UMAP_1_20220215', 'UMAP_2_20220215', 'UMAP_1_20230508', 'UMAP_2_20230508', 'type', 'nCount_RNA3', 'nFeature_RNA3'
    var: 'features'

In [9]:
# Remove tonsil atlas bad genes
tonsil_atlas_adata_specific_genes_to_remove = list(
    set(tonsil_atlas_adata.var_names) - set(tonsil_adata.var_names))
tonsil_atlas_adata = (
    tonsil_atlas_adata[:, ~tonsil_atlas_adata.var_names.isin(
        tonsil_atlas_adata_specific_genes_to_remove)].copy())
tonsil_atlas_adata

AnnData object with n_obs × n_vars = 462352 × 32577
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'barcode', 'donor_id', 'gem_id', 'library_name', 'assay', 'sex', 'age', 'age_group', 'hospital', 'cohort_type', 'cause_for_tonsillectomy', 'is_hashed', 'preservation', 'pct_mt', 'pct_ribosomal', 'pDNN_hashing', 'pDNN_scrublet', 'pDNN_union', 'scrublet_doublet_scores', 'S.Score', 'G2M.Score', 'Phase', 'scrublet_predicted_doublet', 'doublet_score_scDblFinder', 'annotation_level_1', 'annotation_level_1_probability', 'annotation_figure_1', 'annotation_20220215', 'annotation_20220619', 'annotation_20230508', 'annotation_20230508_probability', 'UMAP_1_level_1', 'UMAP_2_level_1', 'UMAP_1_20220215', 'UMAP_2_20220215', 'UMAP_1_20230508', 'UMAP_2_20230508', 'type', 'nCount_RNA3', 'nFeature_RNA3'
    var: 'features'

## Remove the meta-data or precalculated columns that are not required here

In [10]:
remove_obs_keys = [
    'pct_mt', 'pct_ribosomal', 'pDNN_hashing', 'pDNN_scrublet', 'pDNN_union',
    'scrublet_doublet_scores', 'S.Score', 'G2M.Score', 'Phase',
    'scrublet_predicted_doublet', 'doublet_score_scDblFinder', 'orig.ident',
    'nCount_RNA', 'nFeature_RNA', 'nCount_RNA3', 'nFeature_RNA3']

tonsil_atlas_obs = tonsil_atlas_adata.obs.copy()
tonsil_atlas_obs = tonsil_atlas_obs.drop(remove_obs_keys, axis=1)

In [11]:
tonsil_atlas_adata.obs = tonsil_atlas_obs.copy()

In [12]:
tonsil_atlas_adata.obs["sample_type"] = (
    tonsil_atlas_adata.obs["cause_for_tonsillectomy"].copy()).astype("category")

In [13]:
tonsil_adata.obs["study_category"] = "disco_tonsil"
blood_adata.obs["study_category"] = "disco_blood"
tonsil_atlas_adata.obs["study_category"] = "tonsil_atlas"

/scratch/local/jobs/166190/ipykernel_1135831/4153881906.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  tonsil_adata.obs["study_category"] = "disco_tonsil"


# Concatenate multiple adata objects

In [14]:
min_counts_cells = 100
min_counts_genes = 100

In [15]:
%%time
adatas = []
adatas.append(blood_adata)
adatas.append(tonsil_atlas_adata)
adatas.append(tonsil_adata)


# Concatenate all the objects
adata = ad.concat(adatas, join="outer", uns_merge="same", merge="same")

print(f"Full tonsil Adata Count matrix shape: {blood_adata.shape}")
print(f"Full tonsil Adata Count matrix shape: {tonsil_atlas_adata.shape}")
print(f"Full tonsil Adata Count matrix shape: {tonsil_adata.shape}")
adata = adata[:, adata.X.sum(0) > min_counts_genes].copy()
print(f"Full Adata Count matrix shape after gene prefiltering: {adata.shape}")

adata

/ssd/projects/Proximap/mambaforge/scanpy_stable_25_04_latest/lib/python3.12/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(


Full tonsil Adata Count matrix shape: (169686, 33538)
Full tonsil Adata Count matrix shape: (462352, 32577)
Full tonsil Adata Count matrix shape: (92114, 32577)
Full Adata Count matrix shape after gene prefiltering: (724152, 23455)
CPU times: user 19.3 s, sys: 8.75 s, total: 28.1 s
Wall time: 28.1 s


AnnData object with n_obs × n_vars = 724152 × 23455
    obs: 'orig.ident', 'ncount_rna', 'nfeature_rna', 'sample', 'sample_id', 'project_id', 'sample_type', 'disease', 'tissue', 'platform', 'disease_subtype', 'time_point', 'subject_id', 'age', 'gender', 'race', 'infection', 'disease_stage', 'mutation', 'predicted_cell_type', 'ct', 'sub.cluster', 'n_genes', 'n_counts', 'pct_MT', 'pct_RBS', 'pct_MTRBS', 'pct_PSEUDO', 'pct_HG', 'pct_gtex', 'pct_hpa', 'pct_hrt', 'pct_cellminer', 'pct_klijn', 'pct_IG_C_gene', 'pct_IG_C_pseudogene', 'pct_IG_J_gene', 'pct_IG_V_gene', 'pct_IG_V_pseudogene', 'pct_TR_C_gene', 'pct_TR_J_gene', 'pct_TR_J_pseudogene', 'pct_TR_V_gene', 'pct_TR_V_pseudogene', 'pct_lncRNA', 'pct_miRNA', 'pct_protein_coding', 'pct_ribozyme', 'pct_snRNA', 'pct_snoRNA', 'pct_pseudogenes_hg', 'n_unique', 'study_category', 'barcode', 'donor_id', 'gem_id', 'library_name', 'assay', 'sex', 'age_group', 'hospital', 'cohort_type', 'cause_for_tonsillectomy', 'is_hashed', 'preservation', 'annotat

## DISCO mapping

In [16]:
# One time use only, sorry for the missing docs
class CellTree:
    def __init__(self):
        self.children = {}
        self.parent = {}
        self.root = None
        self.special_cells = []
    
    def add_cell(self, cell_type, parent):
        if cell_type in self.parent:
            raise ValueError(f"Cell type '{cell_type}' already exists.")
        
        if parent not in self.children:
            self.children[parent] = []
        
        self.children[parent].append(cell_type)
        self.parent[cell_type] = parent
        
        if parent is None:
            if self.root is not None:
                raise ValueError("Tree can have only one root.")
            self.root = cell_type
    
    def find_root(self):
        if self.root:
            return self.root
        
        all_cells = set(self.parent.keys())
        all_parents = set(self.parent.values())
        
        roots = all_parents - all_cells
        if len(roots) != 1:
            raise ValueError("Tree must have exactly one root.")
        
        self.root = roots.pop()
        return self.root
    
    def find_cell_n_away_from_cell(self, start_cell_type, n, ignore=True):
        if start_cell_type not in self.parent:
            raise ValueError(f"Cell type '{start_cell_type}' not found in the tree.")
        
        current_cell = start_cell_type
        for _ in range(n):
            if current_cell not in self.parent or self.parent[current_cell] is None:
                if not ignore:
                    raise ValueError(f"Cannot find a cell type {n} steps away from '{start_cell_type}' starting from the root.")
                else:
                    return start_cell_type
            
            current_cell = self.parent[current_cell]
        
        return current_cell

    def is_connected(self):
        if self.root is None:
            return False
    
        visited = set()
        stack = [self.root]
    
        while stack:
            node = stack.pop()
            if node not in visited:
                visited.add(node)
                if node in self.children:
                    stack.extend(self.children[node])
        
        # Check if all nodes are visited
        all_nodes = set(self.parent.keys()).union(set(node for children in self.children.values() for node in children))
        all_nodes.add(self.root)  # Include the root node
        return visited == all_nodes

    def find_cell_n_away_from_root(self, start_cell_type, n):
        if self.root is None:
            raise ValueError("The tree does not have a root.")
        
        # Find the path from the root to the start_cell_type
        def find_path(node, target, path):
            path.append(node)
            if node == target:
                return True
            if node in self.children:
                for child in self.children[node]:
                    if find_path(child, target, path):
                        return True
            path.pop()
            return False
        
        path = []
        if not find_path(self.root, start_cell_type, path):
            raise ValueError(f"Cell type '{start_cell_type}' not found in the tree.")
        
        # If n is 0, return the root
        if n == 0:
            return self.root
        
        # Traverse the path to find the nth node from the root
        if n < len(path):
            return path[n]
        else:
            raise ValueError(f"Cannot find a cell type {n} steps away from '{start_cell_type}' starting from the root.")

    def subtree_up_to_depth_nested_dict(self, n):
        if self.root is None:
            raise ValueError("The tree does not have a root.")
        
        def traverse(node, depth):
            if depth > n:
                return None
            subtree = {node: []}
            if node in self.children:
                for child in self.children[node]:
                    child_subtree = traverse(child, depth + 1)
                    if child_subtree is not None:
                        subtree[node].append(child_subtree)
            return subtree
        
        return traverse(self.root, 0)

    def subtree_up_to_depth_df(self, n):
        if self.root is None:
            raise ValueError("The tree does not have a root.")
        
        def traverse(node, depth):
            if depth > n:
                return None
            
            subtree_data = []
            if node in self.children:
                for child in self.children[node]:
                    child_subtree = traverse(child, depth + 1)
                    if child_subtree is not None:
                        subtree_data.extend(child_subtree)
            
            subtree_data.append({'cell_type': node, 'parent': self.parent.get(node)})
            return subtree_data if subtree_data else None
        
        subtree_data = traverse(self.root, 0)
        
        if subtree_data:
            subtree_df = pd.DataFrame(subtree_data)
            return subtree_df[['cell_type', 'parent']]
        else:
            return pd.DataFrame(columns=['cell_type', 'parent'])

    def mark_special_cells(self, special_cells):
        for cell in special_cells:
            if cell not in self.parent:
                raise ValueError(f"Cell type '{cell}' not found in the tree.")
            self.special_cells.append(cell)
    
    def find_closest_special_cell_on_path(self, cell_type):
        if self.root is None:
            raise ValueError("The tree does not have a root.")
        
        if cell_type not in self.parent:
            raise ValueError(f"Cell type '{cell_type}' not found in the tree.")
        
        path_to_root = []
        
        # Find the path from the cell_type to the root
        def find_path_to_root(node):
            path_to_root.append(node)
            if node == self.root:
                return True
            if node in self.parent:
                return find_path_to_root(self.parent[node])
            return False
        
        find_path_to_root(cell_type)
        
        # Find the closest special cell on the path to the root
        for node in path_to_root:
            if node in self.special_cells:
                return node
        
        return None

    def print_path_to_root(self, cell_type):
        if self.root is None:
            raise ValueError("The tree does not have a root.")
        
        if cell_type not in self.parent:
            raise ValueError(f"Cell type '{cell_type}' not found in the tree.")
        
        path_to_root = []
        
        # Find the path from the cell_type to the root
        def find_path_to_root(node):
            path_to_root.append(node)
            if node == self.root:
                return True
            if node in self.parent:
                return find_path_to_root(self.parent[node])
            return False
        
        find_path_to_root(cell_type)
        
        # Print the path to the root
        path_str = " -> ".join(reversed(path_to_root))
        print(f"Path to root from '{cell_type}': {path_str}")

    def find_closest_cell_type(self, target_cell_type):
        if target_cell_type in self.parent:
            return target_cell_type
        
        # Implementing Smith-Waterman-like search as an example
        def smith_waterman(s1, s2):
            # Perform Smith-Waterman alignment here
            # This is a simplified placeholder
            return difflib.SequenceMatcher(None, s1.lower(), s2.lower()).ratio()
        
        # Fallback search: Find the closest cell type based on some criteria
        closest_cell_type = None
        closest_score = -1
        
        for cell_type in self.parent.keys():
            score = smith_waterman(target_cell_type, cell_type)
            if score > closest_score:
                closest_cell_type = cell_type
                closest_score = score
        
        return closest_cell_type

    def print_disconnected_branches(self):
        if self.is_connected():
            return
        
        print("Disconnected branches:")
        visited = set()
        for parent, children in self.children.items():
            visited.add(parent)
            for child in children:
                visited.add(child)
        
        for cell_type in self.parent:
            if cell_type not in visited:
                print(f"{cell_type} -> {self.parent[cell_type]}")


# Test for the CellTree
if False:
    # Example usage:
    # Create a tree
    tree = CellTree()
    tree.add_cell('A', None)  # A is the root
    tree.add_cell('B', 'A')
    tree.add_cell('C', 'A')
    tree.add_cell('D', 'B')
    tree.add_cell('E', 'C')
    
    # Find the cell type that is 2 steps away from 'D'
    try:
        result = tree.find_cell_n_away_from_root('D', 2)
        print(f"The cell type 2 steps away from 'D' is: {result}")
    except ValueError as e:
        print(e)
    
    tree.is_connected(), tree.find_cell_n_away_from_cell('E', 0), tree.find_cell_n_away_from_root('E', 1)
    tree.subtree_up_to_depth_df(2)


ct_tree = pd.read_csv("../data/disco_ct_tree.csv")
# duplicated line removal 
ct_tree = ct_tree.drop(index=450)
# Fix ring
ct_tree.loc[ct_tree["Cell Type"] == "Memory B cell precursor", "Parent"] = "Memory B cell"
ct_tree.loc[ct_tree["Cell Type"] == "G2/M phase DZ GC B cell", "Parent"] = "Cycling DZ GC B cell"
 

tree = CellTree()
for ct, parent in ct_tree[["Cell Type", "Parent"]].values:
    tree.add_cell(ct, parent)

tree.find_root()
# Setup the nodes (cell types) to report 
special_cells = [
        # First layer
        'Doublet like cell',
        'Embryonic cell',
        'Endothelial cell', 
        'Epithelial cell',
        'Fibroblast', 'Germ cell',
        'Glial cell', 'Immune cell', 'Muscle cell', 'Neuron',
        'Perivascular cell',
        # Lymphocytes
        'Lymphoid cell', "B cell", "T/NK cell",
        # Myeloyd
        'Myeloid cell', 'Dendritic cell', 'Granulocyte', 'Macrophage', 'Monocyte', 'Osteoclast',
        # Brain
        'Glial cell of brain', 'Glial cell of intestine', 'Muller glia', 'Schwann cell',
        'Neuron of brain', 'Neuron of intestine', 'Neuron of retina',
        # Epithelial cells
        'Endocrine cell',
        'Mesothelial cell',
        ]

special_epithelial_cells = [
        'Epithelial cell of adrenal gland',
        'Epithelial cell of bladder',
        'Epithelial cell of breast',
        'Epithelial cell of eye',
        'Epithelial cell of gingiva',
        'Epithelial cell of intestine',
        'Epithelial cell of kidney',
        'Epithelial cell of liver',
        'Epithelial cell of lung',
        'Epithelial cell of ovary',
        'Epithelial cell of pancreas',
        'Epithelial cell of skin',
        'Epithelial cell of stomach',
        'Epithelial cell of testis',
        'Epithelial cell of thymus',
        'Epithelial cell of tonsil',
        'PDAC related epithelial cell',
        'Cycling epithelial cell',]

# special_cells = special_cells + special_epithelial_cells

# find children if you want to add more
# tree.children['Myeloid cell']
# or search the database NOTE: Lowercase search if not desired remove the .lower():
# [x for x in ct_tree["Cell Type"].unique() if "mye" in x.lower()]

tree.mark_special_cells(special_cells)

tree.is_connected(), tree.root

(True, 'Cell Type')

## Manually curated database cell-types to the DISCO cell-type tree cells

In [17]:
map_atlas_ct_to_ct_tree = {
    "ADAMDEC1 fibroblast": "Fibroblast",
    "ANGPTL7+ fibroblast": "Fibroblast",
    "APOD fibroblast": "APOD+PTGDS+ fibroblast",
    "Agonist T": "Agonist T cell",
    "C7 fibroblast": "Fibroblast",
    "CCL11 fibroblast": "Fibroblast",
    "AREG NK": "NK cell",
    "BGN EC": "BGN+ EC",
    "Breast basal cell": "Epithelial cell of breast",
    "C3 fibroblast": "Fibroblast",
    "CD16 NK": "CD16 NK cell",
    "CD4 T": "CD4 T cell",
    "CD56 NK": "CD56 NK cell",
    "CD8 T": "CD8 T cell",
    "CD8aa+ T": "CD8aa+ T cell",
    "CFD fibroblast": "CFD+MGP+ fibroblast", 
    "COL11A1 fibroblast": "CCL19+APOE+ fibroblast",
    "CXCL+ fibroblast": "Fibroblast",
    "CXCL13 exhausted CD8 T": "CXCL13 exhausted CD8 T cell",
    "CXCL13 fibroblast": "Fibroblast",
    "CXCL14 fibroblast": "CXCL14+C7+ fibroblast",
    "Ciliated cell": "Multiciliated cell",
    "Class switch memory B": "Class-switched memory B cell",
    "Collagen+ fibroblast": "Collagen+APOD+PTGDS+ fibroblast",
    "Complement factor+ fibroblast": "Fibroblast",
    "DZ/LZ B": "DZ/LZ B cell",
    "Differentiating Treg": "Differentiating Treg cell",
    "Double negative T": "Double negative T cell",
    "Double positive T": "Double positive T cell",
    "F13A1 fibroblast": "Fibroblast",
    "F3 fibroblast": "Fibroblast",
    "FCRL4 memory B": "FCRL4 memory B cell",
    "FRZB fibroblast": "Fibroblast",
    "Fetal MHC- Collagen+ arterial EC": "Fetal MHC- arterial EC",
    "Fetal MHC- Collagen+ capillary EC": "Fetal MHC- capillary EC",
    "Fetal MHC- Collagen+ venous EC": "Fetal MHC- venous EC",
    "G2M DZ B": "G2/M phase DZ GC B cell",
    "GABRP endometrium": "GABRP+ epithelial cell",
    "GC committed cell": "Committed GC B cell",
    "GZMB CD8 T": "GZMB CD8 T cell",
    "GZMK CD8 T": "GZMK CD8 T cell",
    "Glia": "Glial cell",
    "HAND1 mesoderm": "Mesodermal cell",
    "Histone high S phase DZ B": "Histone high S phase DZ GC B cell",
    "IFN responsed trophoblast": "INF-activated trophoblast",
    "IGFBP1 fibroblast": "Fibroblast",
    "INF responsed CD56 NK": "INF-activated CD56 NK cell",
    "INF responsed CD8 T": "INF-activated CD8 T cell",
    "INF responsed T": "INF-activated T cell",
    "INF responsed macrophage": "INF-activated macrophage",
    "INF responsed naive B": "INF-activated naive B cell",
    "ITGAX+ macrophage": "Macrophage",
    "Immune cell recruitment pancreatic ductal cell": "Pancreatic ductal cell",
    "KCNN3 fibroblast": "Fibroblast",
    "KLRB1 CD8 T": "KLRB1 CD8 T cell",
    "KLRB1 cytotoxic CD4 T": "T cell",
    "LUCAT1 fibroblast": "Fibroblast",
    "LZ B": "DZ/LZ B cell",
    "Lactoferrin+ macrophage": "Macrophage",
    "Luminal cell": "Mammary luminal cell",
    "MAIT": "MAIT cell",
    "MHC class II high monocyte/DC/macrophage": "MHCII high CD14 monocyte",
    "MMP1 fibroblast": "Fibroblast",
    "Memory B": "Memory B cell",
    "Memory B precursor": "Memory B cell precursor",
    "Memory CD4 T": "Memory CD4 T cell",
    "Memory CD8 T": "Memory CD8 T cell",
    "Metallothionein+ Collagen- fibroblast": "Collagen+APOD+PTGDS+ fibroblast",
    "NK": "NK cell",
    "Naive B": "Naive B cell",
    "Naive CD4 T": "Naive CD4 T cell",
    "Naive CD8 T": "Naive CD8 T cell",
    "Nonproliferative GC B": "Non-cycling GC B cell",
    "Non-class switch memory B": "Unswitched memory B cell",
    "Ovarian cancer enriched S100A9 MUC16 epithelial cell": "Ovarian cancer enriched S100A9+MUC16+ epithelial cell",
    "Ovarian cancer enriched proliferation epithelial cell": "Ovarian cancer enriched cycling epithelial cell",
    "Ovarian cancer specific LGR5 MUC16 epithelial cell": "Ovarian cancer specific LGR5+MUC16+ epithelial cell",
    "Ovarian cancer specific carbonic anhydrase+ fibroblast": "Ovarian cancer specific EMT cell",
    "PIF1+ G2M DZ B": "PIF1+ G2/M phase DZ GC B cell",
    "PLA2G2A fibroblast": "Fibroblast",
    "PLEKHG2 fibroblast": "Fibroblast",
    "POSTN Fibroblast": "Fibroblast",
    "POSTN fibroblast": "Fibroblast",
    "Placenta CAMP+ antimicrobial monocyte": "Placenta CAMP+ monocyte",
    "Placenta defensin+ antimicrobial monocyte": "Placenta defensin+ monocyte",
    "Plasma Cell": "Plasma cell",
    "Pre-B cell": "PreB cell",
    "Pre-GC B": "Pre-GC B cell",
    "Pro-B cell": "ProB cell",
    "GC B": "Non-cycling GC B cell",
    "Proliferation B": "B cell",
    "Proliferation CXCL13 exhausted CD8 T": "Cycling CXCL13 exhausted CD8 T cell",
    "Proliferation CXCL14 fibroblast": "Fibroblast",
    "Proliferation DN/DP T": "Cycling DN/DP T cell",
    "Proliferation EC": "Cycling EC",
    "Proliferation GC B": "Cycling GC B cell",
    "Proliferation KRT14 basal cell": "Cycling KRT14 basal cell",
    "Proliferation Leydig cell": "Cycling Leydig cell",
    "Proliferation PDAC specific cell": "Cycling PDAC specific ductal cell",
    "Proliferation Sertoli cell": "Cycling Sertoli cell",
    "Proliferation T": "Cycling T cell",
    "Proliferation T cell": "Cycling T cell",
    "Proliferation T/NK": "Cycling T/NK cell",
    "Proliferation extravillous trophoblast": "Cycling extravillous trophoblast",
    "Proliferation fibroblast": "Fibroblast",
    "Proliferation glia": "Cycling glial cell",
    "Proliferation lactocyte": "Cycling lactocyte",
    "Proliferation macrophage": "Cycling macrophage",
    "Proliferation mast cell": "Cycling mast cell",
    "Proliferation pericyte": "Pericyte",
    "Proliferation spermatogonial stem cell": "Cycling spermatogonial stem cell",
    "Proliferation thymic epithelial cell": "Cycling thymic epithelial cell",
    "Proliferation villous cytotrophoblast": "Cycling villous cytotrophoblast",
    "S phase DZ B": "S phase DZ GC B cell",
    "SLC13A4+ Syncytiotrophoblast": "SLC13A4+ syncytiotrophoblast",
    "SLC2A1 macrophage": "Macrophage",
    "SPARCL1 fibroblast": "Fibroblast",
    "SPP1 Macrophage": "SPP1 macrophage",
    "SRGN arterial EC": "Arterial EC",
    "Smooth muscle cell": "Vascular smooth muscle cell",
    "Tfh": "Tfh cell",
    "Tonsillar crypt": "Tonsillar crypt epithelial cell",
    "Treg": "Treg cell",
    "Tuft Cell": "Tuft cell",
    "Unknown fibroblast": "Fibroblast",
    "XCL1 NK": "XCL1 NK cell",
    "ZEB2 mesoderm": "Mesodermal cell",
    "abT (entry)": "abT (entry) cell",
    "cDC": "Dendritic cell",
    "ductal/EC doublet like cell": "Ductal/EC doublet like cell",
    "gdT": "Gamma delta T cell",
    "HSC": "Hematopoietic stem cell",
    "ACTA2+ Arterial EC": "ACTA2+ arterial EC",
    "ACTA2+ Capillary EC": "ACTA2+ capillary EC",
    "ADAMDEC1+CXCL14+ fibroblast": "ADAMDEC1+ADAM28+ fibroblast",
    "AFP epithelial cell": "AFP+ epithelial cell",
    "ANGPTL7+NRP2+ fibroblast": "Fibroblast",
    "APCDD1 fibroblast": "CD9+APCDD1+ fibroblast",
    "APOE fibroblast": "CCL19+APOE+ fibroblast",
    "C2orf40 fibroblast": "Fibroblast",
    "Adipose ITLN1 fibroblast": "Fibroblast",
    "C1QTNF3+SHISA3+COL14A1+ fibroblast": "Fibroblast",
    "C7+ fibroblast": "Fibroblast",
    "CCL19/21 fibroblast": "CCL19+APOE+ fibroblast",
    "CCL2+SFRP2+ fibroblast": "BNC2+ZFPM2+ fibroblast",
    "CD14+MHCIIhigh monocyte": "MHCII high CD14 monocyte",
    "CD14+MHCIIlow monocyte": "MHCII low CD14 monocyte",
    "CD16+ NK": "CD16 NK cell",
    "CD16+ monocyte": "CD16 monocyte",
    "CD55+SEMA3C+ fibroblast": "CFD+MGP+ fibroblast",
    "CD56+ NK": "CD56 NK cell",
    "CDCA7 proliferation fibroblast": "Fibroblast",
    "CDP": "Common dendritic progenitor",
    "CLDN3 epithelial cell": "CLDN3+ epithelial cell",
    "CLP": "Common lymphoid progenitor",
    "CLU+ fibroblast": "Fibroblast",
    "CMP": "Common myeloid progenitor",
    "COCH fibroblast": "COCH+TNN+ fibroblast",
    "CRABP1 mesothelial cell": "CRABP1+ mesothelial cell",
    "CXCL1/2/3 fibroblast": "Fibroblast",
    "CXCL14+POSTN+ fibroblast": "CXCL14+C7+ fibroblast",
    "Cycling fibroblast": "Fibroblast",
    "Cytotoxic CD8 T cell": "CD8 T cell",
    "DIAPH3+ macrophage": "Macrophage",
    "Effector/Memory CD4 T cell": "Memory CD4 T cell",
    "Erythrocyte": "Red blood cell",
    "Eye melanocyte": "Melanocyte",
    "FBLN2+APOD+ fibroblast": "Fibroblast",
    "FGFBP2 fibroblast": "FGFBP2+OLFML2A+ fibroblast",
    "Fibroblast/Granulosa doublet like cell": "Doublet like cell",
    "G0S2+PPP1R14A+ fibroblast": "Fibroblast",
    "GATA4 mesothelial cell": "GATA4+ mesothelial cell",
    "GMP": "Granulocyte-monocyte progenitor",
    "GREB1 fibroblast": "Fibroblast",
    "GZMB+ CD8 T cell": "GZMB CD8 T cell",
    "GZMK NK": "NK cell",
    "GZMK cytotoxic CD4 T": "CD4 T cell",
    "GZMK+ CD8 T cell": "GZMK CD8 T cell",
    "Gamma-delta T cell": "Gamma delta T cell",
    "IFN-activated T cell": "INF-activated T cell",
    "IGFBP6+APOD+ fibroblast": "MFAP5+IGFBP6+ fibroblast",
    "IGFBP6+SFRP4+ fibroblast": "MFAP5+IGFBP6+ fibroblast",
    "LYVE1+ macrophage": "LYVE1 macrophage",
    "MEP": "Megakaryocyte–erythroid progenitor",
    "MET epithelial cell": "Epithelial cell",
    "MKI67 proliferation fibroblast": "Fibroblast",
    "MPP": "Multipotent progenitor",
    "MT2A granulosa cell": "MT2A+ granulosa cell",
    "MTRNR2L1 B": "MTRNR2L1 B cell",
    "Metallothionein smooth muscle cell": "Vascular smooth muscle cell",
    "Metallothionein+ fibroblast": "Fibroblast",
    "Monocyte/DC": "Monocyte",
    "NR4A3 fibroblast": "Fibroblast",
    "OSR1 granulosa cell": "OSR1+ granulosa cell",
    "PAX8 granulosa cell": "PAX8+ granulosa cell",
    "PCOLCE2 fibroblast": "Fibroblast",
    "PDGFRA+BMP4+ fibroblast": "Fibroblast",
    "PLA2G2A+IGFBP4+APOC1+ fibroblast": "MFAP5+IGFBP6+ fibroblast",
    "PLAC9 fibroblast": "Fibroblast",
    "POSTN+ fibroblast": "Fibroblast",
    "PTGDS+CXCL14+ fibroblast": "CXCL14+C7+ fibroblast",
    "Proliferation immune cell": "Immune cell",
    "Proliferation keratinocyte": "Cycling keratinocyte",
    "Proliferation mesothelial cell": "Cycling mesothelial cell",
    "Proliferation myeloid cell": "Cycling myeloid cell",
    "Proliferation perivascular cell": "Cycling perivascular cell",
    "Proliferation primordial germ cell": "Cycling primordial germ cell",
    "RNASE1 fibroblast": "Fibroblast",
    "Renal epithelial cell - Loop of Henle": "Renal epithelial cell (Loop of Henle)",
    "Renal epithelial cell - distal tubules": "Renal epithelial cell (distal tubule)",
    "Renal epithelial cell - proximal tubules": "Renal epithelial cell (proximal tubule)",
    "Retinal ganglion cell": "Ganglion cell",
    "S100A+ preNeutrophil (cycling)": "S100A+ preNeutrophil",
    "SCN7A fibroblast": "Fibroblast",
    "SLAMF1+XCL- ILC": "Lymphoid cell",
    "SLAMF1-XCL+ ILC": "Lymphoid cell",
    "SPRR2F pericyte": "SPRR2F+ pericyte",
    "T/NK": "T/NK cell",
    "T/NK cell (cycling)": "T/NK cell",
    "TAC1 granulosa cell": "TAC1+ granulosa cell",
    "TCIM+NRG1+ fibroblast": "Fibroblast",
    "TFF1 ductal cell": "TFF1+ ductal cell",
    "THY1+ fibroblast": "Fibroblast",
    "TNC fibroblast": "Fibroblast",
    "TUBA1A ductal cell": "TUBA1A+ ductal cell",
    "WISP2 fibroblast": "Fibroblast",
    "cDC2 (cycling)": "Cycling cDC2",
    "memory B": "Memory B cell",
    "memory CD4 T": "Memory CD4 T cell",
    "naive B": "Naive B cell",
    "naive CD4 T": "Naive CD4 T cell",
    "naive CD8 T": "Naive CD8 T cell",
    "neutrophil": "Neutrophil",
    "plasma cell": "Plasma cell",
    "preB cell": "PreB cell",
    "preB cell (cycling)": "PreB cell",
    "proB cell": "ProB cell",
    "proB cell (cycling)": "ProB cell",
    "proliferation monocyte": "Cycling monocyte",
    "proliferation T/NK": "Cycling T/NK cell",
    "pre-pDC (lymphoid origin)": "Lymphoid pre-pDC",
    "pre-pDC (myeloid origin)": "Myeloid pre-pDC",
    # Unknows:
    "THY1+ fibroblast-like pericyte": "Pericyte",
    "Squamous epithelial cell": "Epithelial cell",
    "Endothelial‑mesenchymal transition EC": "Epithelial cell",
    "platelet": "Megakaryocyte",
    "Gastric mucus-secreting cell": "Surface mucous cell",
    "Merkel cell carcinoma enriched epithelial cell": "Merkel cell",
    "Platelet": "Megakaryocyte",
    # Artifacts
    "PAEP endometrium": "unknown",
    "Proliferation unknown": "unknown",    
}

adata.obs["ct_fixed"] = adata.obs["ct"].replace(map_atlas_ct_to_ct_tree)

/scratch/local/jobs/166190/ipykernel_1135831/3156871855.py:257: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs["ct_fixed"] = adata.obs["ct"].replace(map_atlas_ct_to_ct_tree)


## DISCO cell-type tree cells

In [18]:
broad_map = {
    # --- B lineage ---
    "NBC": "B cell",
    "csMBC": "B cell",
    "ncsMBC": "B cell",
    "LZ_DZ transition": "B cell",
    "NBC early activation": "B cell",
    "GC-commited NBC": "B cell",
    "Early GC-commited NBC": "B cell",
    "NBC IFN-activated": "B cell",
    "NBC CD229+": "B cell",
    "Proliferative NBC": "B cell",
    "Early MBC": "B cell",
    "Precursor MBCs": "B cell",
    "MBC FCRL5+": "B cell",
    "MBC derived early PC precursor": "B cell",
    "MBC derived IgA+ PC": "B cell",
    "MBC derived PC precursor": "B cell",
    "MBC derived IgG+ PC": "B cell",
    "Reactivated proliferative MBCs": "B cell",
    "csMBC FCRL4/5+": "B cell",
    "ncsMBC FCRL4/5+": "B cell",
    "GC DZ Noproli": "B cell",

    # --- GC B cells ---
    "DZ_LZ transition": "B cell",
    "DZ non proliferative": "B cell",
    "DZ late Sphase": "B cell",
    "DZ early Sphase": "B cell",
    "DZ late G2Mphase": "B cell",
    "DZ early G2Mphase": "B cell",
    "DZ cell cycle exit": "B cell",
    "LZ": "B cell",
    "LZ_DZ reentry commitment": "B cell",
    "LZ proliferative": "B cell",
    "preGC": "B cell",

    # --- Tfh / T cells ---
    "Tfh-LZ-GC": "T/NK cell",
    "CM PreTfh": "T/NK cell",
    "CM Pre-non-Tfh": "T/NK cell",
    "GC-Tfh-SAP": "T/NK cell",
    "Tfh-Mem": "T/NK cell",
    "Tfr": "T/NK cell",
    "Tfh T:B border": "T/NK cell",
    "GC-Tfh-OX40": "T/NK cell",
    "Eff-Tregs": "T/NK cell",
    "Eff-Tregs-IL32": "T/NK cell",
    "T-helper": "T/NK cell",
    "T-Trans-Mem": "T/NK cell",
    "cycling T": "T/NK cell",
    "DN": "T/NK cell",
    "Naive CD8 T": "T/NK cell",
    "Naive": "T/NK cell",   # ambiguous → Naive B vs T; here likely T
    "CM CD8 T": "T/NK cell",
    "RM CD8 activated T": "T/NK cell",
    "RM CD8 T": "T/NK cell",
    "T-Eff-Mem": "T/NK cell",
    "DC recruiters CD8 T": "T/NK cell",
    "SCM CD8 T": "T/NK cell",
    "ZNF683+ CD8 T": "T/NK cell",
    "EM CD8 T": "T/NK cell",
    "CD8 Tf": "T/NK cell",
    "IFN+ CD8 T": "T/NK cell",
    "TCRVδ+ gd T": "T/NK cell",
    "MAIT/CD161+TRDV2+ gd T-cells": "T/NK cell",

    # --- NK & ILC ---
    "NKp44+ ILC3": "T/NK cell",
    "NKp44- ILC3": "T/NK cell",
    "ILC1": "T/NK cell",
    "CD16-CD56+ NK": "T/NK cell",
    "CD16-CD56dim NK": "T/NK cell",
    "CD16+CD56- NK": "T/NK cell",

    # --- Plasma cells / PCs ---
    "PB": "Plasma cell",
    "PB committed early PC precursor": "Plasma cell",
    "Transitional PB": "Plasma cell",
    "IgG+ PC precursor": "Plasma cell",
    "IgD PC precursor": "Plasma cell",
    "IgM+ PC precursor": "Plasma cell",
    "IgM+ early PC precursor": "Plasma cell",
    "preMature IgM+ PC": "Plasma cell",
    "preMature IgG+ PC": "Plasma cell",
    "Early PC precursor": "Plasma cell",
    "Short lived IgM+ PC": "Plasma cell",
    "PC committed Light Zone GCBC": "Plasma cell",
    "Mature IgA+ PC": "Plasma cell",
    "Mature IgM+ PC": "Plasma cell",
    "Mature IgG+ PC": "Plasma cell",
    "DZ migratory PC precursor": "Plasma cell",

    # --- Dendritic cells ---
    "DC1 precursor": "Dendritic cell",
    "DC1 mature": "Dendritic cell",
    "aDC1": "Dendritic cell",
    "aDC2": "Dendritic cell",
    "aDC3": "Dendritic cell",
    "DC2": "Dendritic cell",
    "DC3": "Dendritic cell",
    "DC4": "Dendritic cell",
    "DC5": "Dendritic cell",
    "IL7R DC": "Dendritic cell",
    "PDC": "Dendritic cell",
    "IFN1+ PDC": "Dendritic cell",

    # --- FDC & stromal ---
    "FDC": "Dendritic cell",
    "cycling FDC": "Dendritic cell",
    "COL27A1+ FDC": "Dendritic cell",
    "CD14+CD55+ FDC": "Dendritic cell",

    # --- Monocytes / macrophages / myeloid ---
    "Monocytes": "Monocyte",
    "M1 Macrophages": "Macrophage",
    "SELENOP Slan-like": "Myeloid cell",
    "C1Q Slan-like": "Myeloid cell",
    "MMP Slan-like": "Myeloid cell",
    "ITGAX Slan-like": "Myeloid cell",

    # --- Granulocytes ---
    "Neutrophils": "Granulocyte",
    "Mast": "Granulocyte",

    # --- Epithelium / stromal ---
    "Crypt": "Epithelial cell",
    "Basal cells": "Epithelial cell",
    "Surface epithelium": "Epithelial cell",
    "FDCSP epithelium": "Epithelial cell",
    "Outer surface": "Epithelial cell",

    # --- Stromal / fibroblast-like ---
    "MRC": "Stromal cell",
    "FRC": "Stromal cell",

    # --- Other / cycling ---
    "Cycling": "Immune cell",
    "VEGFA+": "Immune cell",
    "preB": "B cell",
    "preT": "T/NK cell",

    # --- Unknowns ---
    "unannotated": "unknown",
    "unknown": "unknown",
}

In [19]:
col1 = "ct_fixed"  # Disco cell-type tree cell types finegrained          
col2 = "annotation_20230508"  # Tonsil atlas finegrained column 
unknown = "unknown"  # label you use for unknowns

def map_with_tree(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    return x if x == unknown else tree.find_closest_special_cell_on_path(x)

# Unified Broad categories for the Disco atlasses
adata.obs["ct_broad"] = adata.obs[col1].apply(map_with_tree).astype(str)

# Unified Broad categories for the Tonsil atlasses
tonsil_subsetter = (adata.obs["study_category"] == "tonsil_atlas")

adata.obs.loc[tonsil_subsetter, "ct_broad"] = (
    adata.obs.loc[tonsil_subsetter, col2].map(broad_map))
# Cast the ct_broad as categorical to for the grouping
adata.obs["ct_broad"] = adata.obs["ct_broad"].astype("category")


In [20]:
adata.obs["ct_finegrained_combined"] = adata.obs[col1].astype(str)
adata.obs.loc[tonsil_subsetter, "ct_finegrained_combined"] = (
    adata.obs.loc[tonsil_subsetter, col2])
adata.obs["ct_finegrained_combined"] = adata.obs["ct_finegrained_combined"].astype("category")

# Excourse - Sequencing depth

# What is the recommended sequencing depth for Single Cell 3' and 5' Gene Expression libraries?

**Source:** [10x Genomics Knowledge Base](https://kb.10xgenomics.com/s/article/115002022743-What-is-the-recommended-sequencing-depth-for-Single-Cell-3-and-5-Gene-Expression-libraries)

---

### Detailed Information

**Question:** What is the recommended sequencing depth for Single Cell 3' and 5' Gene Expression libraries?

**Answer:** For new sample types, we recommend sequencing a minimum of **20,000 read pairs/cell** for Single Cell 3' and Single Cell 5' gene expression libraries.

The sequencing depth required for a particular experiment, however, will depend on:

- Sample type (different samples will have more or less RNA per cell)
- The experimental question being addressed.

After sequencing, the **'Sequencing Saturation'** metric reported by Cell Ranger can be used to optimize sequencing depth for specific sample types.

For instance, with **50,000 read pairs/cell** for RNA-rich cells such as cell lines, only **30-50% sequencing saturation** may be reached. However, this may be sufficient to cluster your sub-populations of interest during the analysis.

If you are interested in identifying as many genes as possible in your sample, you may want to sequence deeper to reach around **90% sequencing saturation level.**

For more information, see the *Technical Note Resolving Cell Types as a Function of Read Depth and Cell Number.*

**Products:** Single Cell Gene Expression, Single Cell Immune Profiling


---

In our dataset, the **Tonsil Atlas** shows a maximum mean sequencing depth  
of approximately **25,000 reads per cell**, which is generally sufficient  
for identifying **broad immune lineages**.  

However, this depth is **not optimal for resolving fine subpopulations**,  
as lower sequencing saturation limits the detection of lowly expressed genes  
that are often crucial for distinguishing closely related cell states.  

Therefore, while the overall lineage structure can be trusted,  
the **subpopulation-level resolution in the DISCO dataset** should be  
interpreted with caution.  



In [21]:
vc = adata.obs["ct_finegrained_combined"].value_counts()
vc[vc < 200]

ct_finegrained_combined
cDC2                               189
aDC3                               187
Cycling                            185
Double negative T cell             172
PB committed early PC precursor    155
M1 Macrophages                     155
Early PC precursor                 147
EM CD8 T                           138
Surface epithelium                 134
CD16+CD56- NK                      128
DZ migratory PC precursor          113
Cycling myeloid cell               110
Crypt                              109
Hematopoietic stem cell            108
IgM+ early PC precursor            107
IL7R DC                            106
DC4                                101
MRC                                 95
Transitional PB                     91
IFN1+ PDC                           85
Mast                                80
NKp44- ILC3                         76
Macrophage                          75
preB                                75
Cycling T cell                      73
B

In [22]:
# Remove cell type categories having less than 200 cells,
# because we cannot be certaion about subpopulation annotations for the low sequencing depth data
# and it is not the purpose of the study to evaluate rare cell-types (< 1%)

adata = adata[~adata.obs["ct_finegrained_combined"].isin(vc[vc < 200].index.tolist())]
adata

View of AnnData object with n_obs × n_vars = 720675 × 23455
    obs: 'orig.ident', 'ncount_rna', 'nfeature_rna', 'sample', 'sample_id', 'project_id', 'sample_type', 'disease', 'tissue', 'platform', 'disease_subtype', 'time_point', 'subject_id', 'age', 'gender', 'race', 'infection', 'disease_stage', 'mutation', 'predicted_cell_type', 'ct', 'sub.cluster', 'n_genes', 'n_counts', 'pct_MT', 'pct_RBS', 'pct_MTRBS', 'pct_PSEUDO', 'pct_HG', 'pct_gtex', 'pct_hpa', 'pct_hrt', 'pct_cellminer', 'pct_klijn', 'pct_IG_C_gene', 'pct_IG_C_pseudogene', 'pct_IG_J_gene', 'pct_IG_V_gene', 'pct_IG_V_pseudogene', 'pct_TR_C_gene', 'pct_TR_J_gene', 'pct_TR_J_pseudogene', 'pct_TR_V_gene', 'pct_TR_V_pseudogene', 'pct_lncRNA', 'pct_miRNA', 'pct_protein_coding', 'pct_ribozyme', 'pct_snRNA', 'pct_snoRNA', 'pct_pseudogenes_hg', 'n_unique', 'study_category', 'barcode', 'donor_id', 'gem_id', 'library_name', 'assay', 'sex', 'age_group', 'hospital', 'cohort_type', 'cause_for_tonsillectomy', 'is_hashed', 'preservation', 

# Save the object

In [23]:
# If you merge multiple studies there might be some columns in the adata.obs 
# that have multiple data types (dtypes) (nan is a float, but the column has string values)
# Unfortunately the h5 cannot save mixed dtypes improperly handled,
# this function fixes all these columns to h5 compatible dtypes
adata.obs = sc_utils.convert_mixed_types(adata.obs.copy(), fix_multitype=True)

In [24]:
# This is not part of the pipeline, so we use the adata saving functionality

In [26]:
%%time
adata.write_h5ad("../data/00_tonsil_blood_DISCO_merged.h5ad", compression="lzf")

CPU times: user 43.5 s, sys: 1.93 s, total: 45.4 s
Wall time: 46 s
